# Interactive sales explorer

## Problem statement

Build an interactive exploration of a retail sales dataset: revenue trends, category mix, regional performance, and an animated view of how the mix changed across quarters.

### Pipeline

Problem -> Data -> EDA -> Preprocessing -> Modeling -> Evaluation -> Deployment notes -> Conclusion.

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
np.random.seed(0)
print('matplotlib', matplotlib.__version__)

## Data

Generate a realistic sales table with seasonality and categories.

In [ ]:
rng = np.random.default_rng(7)
dates = pd.date_range('2025-01-01', periods=730, freq='D')
categories = ['electronics', 'grocery', 'apparel', 'home']
rows = []
for cat in categories:
    base = {'electronics': 4000, 'grocery': 2500, 'apparel': 1800, 'home': 2200}[cat]
    season = 1 + 0.3 * np.sin(np.arange(730) / 365 * 2 * np.pi)
    amount = base * season * rng.lognormal(0, 0.25, 730)
    rows.append(pd.DataFrame({'date': dates, 'category': cat, 'revenue': amount.round(2)}))
sales = pd.concat(rows, ignore_index=True)
sales['region'] = rng.choice(['north', 'south', 'east', 'west'], len(sales))
sales['quarter'] = sales['date'].dt.to_period('Q').astype(str)
print(sales.head())
print('rows:', len(sales), '| span:', sales['date'].min().date(), 'to', sales['date'].max().date())

## Plotly availability

Fall back to matplotlib if plotly is not installed.

In [ ]:
try:
    import plotly.express as px
    HAS_PLOTLY = True
    print('plotly available')
except ImportError:
    HAS_PLOTLY = False
    print('plotly not installed; static fallbacks will be used')

## Revenue trend

Monthly revenue with a range slider.

In [ ]:
monthly = (sales.groupby([pd.Grouper(key='date', freq='ME'), 'category'],
                          as_index=False)['revenue'].sum())
print(monthly.head())
if HAS_PLOTLY:
    fig = px.line(monthly, x='date', y='revenue', color='category',
                  title='Monthly revenue by category')
    fig.update_xaxes(rangeslider_visible=True)
    print('traces:', len(fig.data))
else:
    fig, ax = plt.subplots(figsize=(10, 4))
    for cat, grp in monthly.groupby('category'):
        ax.plot(grp['date'], grp['revenue'], label=cat)
    ax.legend()
    ax.set_title('Monthly revenue by category')
    fig.tight_layout()

## Category mix

Share of revenue and how concentrated it is.

In [ ]:
mix = sales.groupby('category', as_index=False)['revenue'].sum()
mix['share'] = (mix['revenue'] / mix['revenue'].sum() * 100).round(1)
print(mix.sort_values('share', ascending=False))
hhi = ((mix['share'] / 100) ** 2).sum()
print(f'Herfindahl concentration index: {hhi:.3f} (1.0 would be a single category)')
if HAS_PLOTLY:
    fig = px.bar(mix.sort_values('share'), x='share', y='category', orientation='h',
                 title='Share of revenue by category', text='share')
    print('bars:', len(mix))

## Regional performance

A matrix of category by region.

In [ ]:
matrix = sales.pivot_table(index='region', columns='category',
                           values='revenue', aggfunc='sum').round(0)
print(matrix)
best = matrix.stack().idxmax()
print('strongest combination:', best)
if HAS_PLOTLY:
    fig = px.imshow(matrix, text_auto='.2s', aspect='auto',
                    title='Revenue by region and category')
    print('heatmap created')
else:
    fig, ax = plt.subplots(figsize=(7, 4))
    sns.heatmap(matrix, annot=True, fmt='.0f', cmap='YlGnBu', ax=ax)
    fig.tight_layout()

## Animated quarterly view

Animation shows how the mix shifted over time.

In [ ]:
quarterly = (sales.groupby(['quarter', 'category', 'region'], as_index=False)['revenue']
             .sum())
print(quarterly.head())
if HAS_PLOTLY:
    fig = px.bar(quarterly, x='category', y='revenue', color='region',
                 animation_frame='quarter', range_y=[0, quarterly['revenue'].max() * 1.1],
                 title='Revenue mix by quarter')
    print('animation frames:', len(fig.frames))
else:
    print(quarterly.pivot_table(index='quarter', columns='category',
                                values='revenue', aggfunc='sum').round(0))

## Outlier review

Flag the days a stakeholder will ask about.

In [ ]:
daily = sales.groupby('date', as_index=False)['revenue'].sum()
mu, sigma = daily['revenue'].mean(), daily['revenue'].std()
daily['z'] = ((daily['revenue'] - mu) / sigma).round(2)
flagged = daily[daily['z'].abs() > 2.5].sort_values('z')
print(f'mean {mu:,.0f} | std {sigma:,.0f} | flagged days: {len(flagged)}')
print(flagged.head(8).to_string(index=False))
fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(daily['date'], daily['revenue'], linewidth=1, alpha=0.7)
ax.scatter(flagged['date'], flagged['revenue'], color='crimson', zorder=3, label='flagged')
ax.set_title('Daily revenue with outliers highlighted')
ax.legend()
fig.tight_layout()

## Export

Write a standalone HTML file for sharing.

In [ ]:
from pathlib import Path
out = Path('artifacts')
out.mkdir(exist_ok=True)
if HAS_PLOTLY:
    fig = px.line(monthly, x='date', y='revenue', color='category',
                  title='Monthly revenue by category')
    path = out / 'sales_explorer.html'
    fig.write_html(path, include_plotlyjs='cdn')
    print('written', path.name, path.stat().st_size, 'bytes')
else:
    monthly.to_csv(out / 'monthly_revenue.csv', index=False)
    print('wrote monthly_revenue.csv as a fallback artefact')

## Conclusion

Write three bullets here after you run the notebook: what the model does well, where it fails, and what you would try with one more week.